# 👑 PRO부동산 장기기억 지식 주입 GGUF 원클릭 빌더
이 노트북은 자동으로 깃허브에서 데이터를 불러와 unsloth/gemma-4-E2B-it 모델 파인튜닝 후 허깅페이스에 `.gguf` 파일로 빌드합니다.

### ⚠️ 실행 전 필수 설정
1. 상단 메뉴의 **런타임 > 런타임 유형 변경**에서 GPU(예: T4, L4, A100 등)를 반드시 선택해 주세요.
2. 구글 코랩 왼쪽 열쇠 모양 아이콘(Secrets)에 **`HF_TOKEN`** 이름으로 대표님의 허깅페이스 Write 토큰을 등록해 주세요.

In [ ]:
# 1. 가상 환경에 필수 라이브러리 설치 (의존성 자동 정렬)
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo


In [ ]:
# 2. 허깅페이스 로그인 및 베이스 모델 로드
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=hf_token)
    print("🔑 HF_TOKEN 코랩 Secrets 연동 로그인 성공!")
except Exception:
    print("ℹ️ 코랩 Secrets에 HF_TOKEN이 없습니다. 아래 수동 로그인 위젯을 통해 로그인해 주세요.")
    from huggingface_hub import notebook_login
    notebook_login()
    hf_token = True

import gc
import torch
from unsloth import FastLanguageModel

print("🔄 [시스템] 베이스 모델 로딩 중...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
    full_finetuning = False,
)

# LoRA (PEFT) 설정
model = FastLanguageModel.get_peft_model(
    model,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    finetune_vision_layers = False,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.0,
    bias = "none",
    random_state = 3407,
)
print("✅ 베이스 모델 및 LoRA 설정 완료!")


In [ ]:
# 3. 지정된 깃허브 저장소에서 데이터셋 원격 호출 및 챗 템플릿 가공
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
import urllib.request
from google.colab import userdata

print("📦 [시스템] 지정된 멀티 깃허브 저장소(hijinjoo2000-prog/maketing)에서 데이터를 원격 호출합니다...")
url = "https://raw.githubusercontent.com/hijinjoo2000-prog/maketing/main/master_dataset.jsonl"
try:
    # Private 저장소일 경우 코랩 Secrets에서 GITHUB_TOKEN을 읽어와 인증합니다.
    gh_token = userdata.get('GITHUB_TOKEN')
    req = urllib.request.Request(url, headers={'Authorization': f'token {gh_token}'})
    print("🔑 GITHUB_TOKEN 보안 인증 연동 성공!")
except Exception:
    req = urllib.request.Request(url)
    print("ℹ️ GITHUB_TOKEN이 Secrets에 없거나 오류가 발생하여 비인증(Public) 호출을 수행합니다.")

try:
    with urllib.request.urlopen(req) as response:
        with open('master_dataset.jsonl', 'wb') as f:
            f.write(response.read())
    ds = load_dataset('json', data_files='master_dataset.jsonl', split='train')
except Exception as e:
    print("❌ 데이터 로드 에러! 저장소가 Private인데 GITHUB_TOKEN이 비어있거나 권한이 없는지 확인하세요.")
    raise e

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def fmt(ex):
    cleaned = []
    for turn in ex["conversations"]:
        role = turn.get("from", turn.get("role", "")).strip().lower()
        val = turn.get("value", turn.get("content", ""))
        standard_role = "assistant" if role in ["model", "assistant", "답변"] else "user"
        cleaned.append({"role": standard_role, "content": val})
    try: 
        return {"text": tokenizer.apply_chat_template(cleaned, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")}
    except Exception as e: 
        print("Formatting error:", e)
        return {"text": ""}

ds = ds.map(fmt, batched=False).filter(lambda x: x["text"] != "")
# conversations 컬럼 제거 -> text 컬럼만 학습
ds = ds.remove_columns([col for col in ds.column_names if col != "text"])
print(f"[시스템] 학습 데이터 준비 완료: {len(ds)}개")


In [ ]:
# 4. SFTTrainer 가상 엔진 조립 및 응답 마스킹
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = "text",
    max_seq_length = 1024,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 1024,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 84,
        learning_rate = 0.00025,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "./outputs",
        save_strategy = "no",
        report_to = "none",
    ),
)

# 🎭 응답만 학습하도록 질문 파트 마스킹 (Loss 안정화)
from unsloth.chat_templates import train_on_responses_only
_t = ds[0]["text"]
_im = "<|turn>user\n" if "<|turn>user" in _t else "<start_of_turn>user\n"
_rm = "<|turn>model\n" if "<|turn>model" in _t else "<start_of_turn>model\n"
trainer = train_on_responses_only(trainer, instruction_part=_im, response_part=_rm)
print(f"✅ 마스킹 마커 자동감지 완료: {_rm.strip()} — 학습 준비 완료!")


In [ ]:
# 5. 파인튜닝 학습 최종 기동
print("🔥 [시스템] 파인튜닝 지식 주입 최종 학습 시작...")
trainer_stats = trainer.train()
print("🎉 학습 완료! 누적 평균 loss:", round(trainer_stats.training_loss, 4))
_last_loss = "알 수 없음"
for log in reversed(trainer.state.log_history):
    if "loss" in log:
        _last_loss = round(log["loss"], 4)
        break
print("📊 최종 단계(마지막 스텝) 실시간 loss:", _last_loss)
print("💡 [중요] \'누적 평균 loss\'는 학습 초반(첫 에포크)의 높은 오차가 누적 합산된 평균값입니다.")
print("💡 모델이 실제 잘 외웠는지는 마지막 스텝 실시간 loss가 Sweet Spot(0.2~0.4)에 도달했는지 확인해 주세요!")


In [ ]:
# 6. 학습 완료 모델 자가진단 추론 테스트
print("🧪 [테스트] 학습이 완료된 모델로 자가 진단 테스트를 가동합니다...")
FastModel.for_inference(model)
def chat(prompt, max_tokens=220):
    msg = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to("cuda")
    if inp["input_ids"][0,0].item() == tokenizer.bos_token_id:
        inp["input_ids"] = inp["input_ids"][:,1:]; inp["attention_mask"] = inp["attention_mask"][:,1:]
    out = model.generate(**inp, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"❓ 질문: {prompt}\n💬 답변: {ans}\n" + "─"*58)

chat("내 사업/지식에 대해 아는 걸 알려줘")
chat("너는 무엇을 도와줄 수 있어?")
chat("내가 인서울 재개발지 어느곳을 정하던지, 최적화 블로그 글을 작성할 수 있니?")


In [ ]:
# 7. GGUF 변환 및 허깅페이스 저장소 자동 업로드
from google.colab import userdata
try: hf_token = userdata.get('HF_TOKEN')
except Exception: hf_token = True

try: del trainer
except: pass
gc.collect(); torch.cuda.empty_cache()

print("📦 [시스템] 지정하신 최종 허깅페이스 창고(seojinju8818/marketing-v10)로 LoRA 어댑터 가중치(.safetensors) 업로드를 시작합니다...")
model.push_to_hub("seojinju8818/marketing-v10", tokenizer = tokenizer, token = hf_token)

print("📦 [시스템] 지정하신 최종 허깅페이스 창고(seojinju8818/marketing-v10)로 GGUF 빌드 및 업로드를 시작합니다...")
model.push_to_hub_gguf("seojinju8818/marketing-v10", tokenizer, quantization_method = "q4_k_m", token = hf_token)
print("🎉 [대성공] LoRA 어댑터(.safetensors) 및 GGUF 모델 빌드/업로드가 완료되었습니다!")
